# Knot Detector CNN - Counting Crossings

TODO: This notebook is designed to run in colab or locally.


## 1) Environment detection, paths, and expected tree

In [7]:

import os, sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)

# Project root
if IN_COLAB:
    PROJ_ROOT = Path("/content/knot-cnn").resolve()
else:
    PROJ_ROOT = Path.cwd()
    if PROJ_ROOT.name == "notebooks":
        PROJ_ROOT = PROJ_ROOT.parent

print("Project root:", PROJ_ROOT)

DATA_ROOT = PROJ_ROOT / "data" / "raw" / "knot-crossings@main"
(DATA_ROOT / "train").mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "test").mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJ_ROOT))
print("sys.path[0]:", sys.path[0])


IN_COLAB: False
Project root: /Users/annedranowski/HQ/🚀 Projects/Knot Detector
sys.path[0]: /Users/annedranowski/HQ/🚀 Projects/Knot Detector


## 2) Colab only - Lightweight installs

In [8]:

if IN_COLAB:
    !pip -q install -U huggingface_hub torchvision matplotlib scikit-learn tqdm
    try:
        import torch, torchvision  # noqa: F401
        print("Using preinstalled torch/torchvision.")
    except Exception:
        !pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cpu


## 3) Pull train/test tarballs from Hugging Face (optional if local data already present)

In [9]:

from huggingface_hub import list_repo_files, hf_hub_download
import tarfile
from pathlib import Path

HF_REPO_ID = "tr33hugg3r/knot-crossings"
HF_REVISION = "main"
DEST = DATA_ROOT

def _extract_tar_gz(tar_path: Path, dest_dir: Path):
    dest_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as t:
        t.extractall(path=dest_dir)

try:
    files = list_repo_files(HF_REPO_ID, revision=HF_REVISION)
    tarballs = [f for f in files if f.endswith(".tar.gz")]
    print("Found tarballs:", tarballs)
    for name in ["train.tar.gz", "test.tar.gz"]:
        found = [f for f in tarballs if f.split("/")[-1] == name]
        if found:
            fp = hf_hub_download(repo_id=HF_REPO_ID, filename=found[0], revision=HF_REVISION)
            fp = Path(fp)
            if name.startswith("train"):
                _extract_tar_gz(fp, DEST / "train")
            else:
                _extract_tar_gz(fp, DEST / "test")
except Exception as e:
    print("HF download step skipped or failed gracefully:", e)

print("Train imgs:", sum(1 for _ in (DEST / "train").rglob("*.*")))
print("Test imgs:",  sum(1 for _ in (DEST / "test").rglob("*.*")))


HF download step skipped or failed gracefully: 404 Client Error. (Request ID: Root=1-68a2d0be-0e6384dd0da59c022fb0a8f1;dc3814f0-54ab-4606-a46c-ed6e400b2b41)

Repository Not Found for url: https://huggingface.co/api/models/tr33hugg3r/knot-crossings/tree/main?recursive=True&expand=False.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Train imgs: 9052
Test imgs: 50


## 4) Our model

In [11]:

# CNN
import torch
from torch import nn

class KnotCNN(nn.Module):
    def __init__(self, output_shape: int):
        super().__init__()

        self.conv_1 = nn.Sequential(
          nn.Conv2d(1, 4, kernel_size=11, stride=1, dilation=2, padding=0),
          nn.BatchNorm2d(4),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2),

          nn.Conv2d(4, 16, kernel_size=5, stride=1, dilation=2, padding=0),
          nn.BatchNorm2d(16),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2),

          nn.Conv2d(16, 64, kernel_size=3, stride=1, dilation=2, padding=0),
          nn.BatchNorm2d(64),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2),

          nn.Conv2d(64, 256, kernel_size=3, stride=1, dilation=2, padding=0),
          nn.BatchNorm2d(256),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2),

          nn.Conv2d(256, 361, kernel_size=3, stride=1, padding=0),
          nn.BatchNorm2d(361),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(361*(12)**2),
            nn.Dropout(p=0.7),
            nn.Linear(in_features=361*(12)**2, out_features=4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(),
            nn.Dropout(p=0.7),
            nn.Linear(in_features=4096, out_features=4096),
            nn.ReLU(),
            nn.Linear(4096, output_shape)
        )

    def forward(self, x: torch.Tensor):
        x = self.conv_1(x)
        x = self.classifier(x)
        return x


## 5) Data pipeline (grayscale, optional invert, ToTensor)

In [12]:

import torch
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

IMG_SIZE = 480  # keep original scale unless you downsize explicitly
BATCH_SIZE = 32

# You were using v2 with grayscale + (optional) invert
data_transform = v2.Compose([
    v2.Grayscale(num_output_channels=1),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    # Uncomment if you want the invert used in your scratch cells
    # v2.functional.invert
])

train_dir = str(DATA_ROOT / "train")
test_dir  = str(DATA_ROOT / "test")

# We keep your original convention: labels from directory names (numbers) but we still
# use MSE w/ regression-style output (rounded for accuracy)
train_data = datasets.ImageFolder(root=train_dir, transform=data_transform)
test_data  = datasets.ImageFolder(root=test_dir,  transform=data_transform)

print("Classes (folder names):", train_data.classes)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=False)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)


DEVICE: cpu
Classes (folder names): ['0', '10', '3', '4', '5', '6', '7', '8', '9']


## 6) Instantiate model & original loss/optimizer/scheduler

In [13]:

import torch
from torch import nn

# Output is a single scalar (regression to crossing count), as in your original notebook
model = KnotCNN(output_shape=1).to(DEVICE)

# Preserve your loss choice
loss_fn = nn.MSELoss()

# Reasonable defaults aligning with your scratch: AdamW + ReduceLROnPlateau
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=2, threshold=1e-3)

def count_parameters(m): 
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("Trainable params:", count_parameters(model))


Trainable params: 230820732


## 7) Training & eval loops (round predictions; same accuracy pattern)

In [14]:

from typing import Callable

def accuracy_fn_regression_round(y_true: torch.Tensor, y_pred_logits: torch.Tensor) -> torch.Tensor:
    # y_true: integer labels (e.g., 0,3,4,...)
    # y_pred_logits: shape [B, 1]; we round to nearest int and compare equality
    y_pred_round = torch.round(y_pred_logits.squeeze(1))
    return (y_pred_round == y_true).float().mean()

def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn: Callable,
               device: torch.device = DEVICE):
    model.train()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for X, y in data_loader:
        X, y = X.to(device), y.to(device).float()  # loss expects float targets
        optimizer.zero_grad()
        y_pred = model(X).squeeze(1)  # [B]
        loss = loss_fn(y_pred, y)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            acc = accuracy_fn(y_true=y, y_pred_logits=y_pred.unsqueeze(1))
        bs = X.size(0)
        total_loss += float(loss) * bs
        total_acc  += float(acc) * bs
        n += bs
    return total_loss / max(n,1), total_acc / max(n,1)

@torch.no_grad()
def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               accuracy_fn: Callable,
               device: torch.device = DEVICE):
    model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for X, y in data_loader:
        X, y = X.to(device), y.to(device).float()
        y_pred = model(X).squeeze(1)
        loss = loss_fn(y_pred, y)
        acc = accuracy_fn(y_true=y, y_pred_logits=y_pred.unsqueeze(1))
        bs = X.size(0)
        total_loss += float(loss) * bs
        total_acc  += float(acc) * bs
        n += bs
    return total_loss / max(n,1), total_acc / max(n,1)


## 8) Run a short training (epochs small by default)

In [16]:
IMG_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 1
MAX_TRAIN_BATCHES = 3
MAX_TEST_BATCHES = 3
num_workers = 0
pin_memory = False

# re-create loaders with new params before training

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_step(model, train_loader, loss_fn, optimizer, accuracy_fn_regression_round, DEVICE)
    te_loss, te_acc = eval_model(model, test_loader, loss_fn, accuracy_fn_regression_round, DEVICE)
    scheduler.step(te_loss)
    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | test loss {te_loss:.4f} acc {te_acc:.3f}")


KeyboardInterrupt: 

## 9) Round-based confusion matrix built from regression outputs

In [ ]:

import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Build mapping from class index -> numeric label (folder names are strings)
# We assume they are numeric like '0','3','4',..., so convert to ints
class_names = train_data.classes
class_ints = [int(c) for c in class_names]

# Build arrays of y_true and y_pred (rounded)
all_y_true, all_y_pred = [], []
model.eval()
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(DEVICE)
        logits = model(X).squeeze(1).cpu().numpy()
        preds = np.rint(logits).astype(int).tolist()
        all_y_pred.extend(preds)
        all_y_true.extend(y.numpy().tolist())

# Convert y_true indices to class integers
idx_to_int = {i: v for i, v in enumerate(class_ints)}
true_vals = [idx_to_int[i] for i in all_y_true]

# Confusion matrix over the numeric labels present
unique_labels = sorted(class_ints)
cm = confusion_matrix(true_vals, all_y_pred, labels=unique_labels)
print(classification_report(true_vals, all_y_pred, labels=unique_labels, digits=3))

fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(cm, interpolation='nearest')
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=range(len(unique_labels)), yticks=range(len(unique_labels)))
ax.set_xticklabels(unique_labels, rotation=45, ha="right")
ax.set_yticklabels(unique_labels)
ax.set_xlabel("Predicted (rounded)")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (Regression → Rounded)")
plt.tight_layout()
plt.show()


## 10) Save artifacts

In [ ]:

from pathlib import Path
from datetime import datetime
ts = datetime.now().strftime("%Y%m%d-%H%M%S")
OUT = (PROJ_ROOT / "outputs")
OUT.mkdir(parents=True, exist_ok=True)

model_path = OUT / f"model_preserved_{ts}.pth"
torch.save(model.state_dict(), model_path)
print("Saved:", model_path)
